#    Libraries

In [ ]:
import cv2
import os
import time
from datetime import datetime
from ultralytics import YOLO 

#    Environment Setup

In [ ]:
os.makedirs("Intruders", exist_ok=True) # Create folder for images if it doesn't exist
print("Loading the model...")
model = YOLO("yolov8n.pt") # Load the smallest and fastest version of YOLOv8
print("Model loaded successfully! Starting the camera... (Press 'Q' to exit)")

#    Camera Initialization

In [ ]:
cap = cv2.VideoCapture(0) # 0 refers to the primary laptop camera

last_save_time = 0
cooldown_seconds = 5 # The model will wait 5 seconds before capturing the same person again

#    Real-time Analysis Loop & Drawing & Decision Making (All in ONE Cell)

In [ ]:
 
# cap.read() ==> This function only captures one Photo at the moment and make two outputs :

#                ret(return) ==> boolean value == True if the cam is open 

#                frame ==> it's an array keeping the pixels that cam cought

# if not ret ==> Safety Condition , if the camera closed or stopped for a reason the loop breaks 

#========================================================================================================================================================================================================

# results = model(frame, classes=[0], conf=0.5, verbose=False) :  frame ==> takes the frame(the pic) and put it in YOLO Nural Net

#                                                                 classes=[0] ==> YOLO Model is trained to detect 80 Different objects , [0] measn selecting the 'COCO' (commom objects in contex)

#                                                                 conf=0.5 ==>   Confidence , the model will only make a decision if he's 50% sure

#                                                                 person_detected = False ==> resets the variable every loop 

#========================================================================================================================================================================================================

# results loop :       x1, y1, x2, y2 ==> coordinates the boxes , turning them to ineger because there's no float pixsels 

#                      box.xyxy[0] ==> 'xyxy' ==> it's a mathimatical form extractes the coordinates of the box's angles (two points , and its enough to make a box) 
#                                       [0] ==> The model is built on pytorch which makes an output in 2D Array , so this will convert it to '[]' 
#                      cv2.rectangle ==> this function draws the box (frame) , note that openCV lib works with BGR not RGB , and the number (2) is the thickness of the line 

#                      cv2.putText ==> FONT_HERSHEY_SIMPLEX ==> font type 
#                                      0.8 ==> font scale
#                                      2   ==> font thickness

#                      person_detected = True == > As we entered the loop thats mean there's a person so we rais a 'True' flag to make a decisions on it later

#========================================================================================================================================================================================================

# Decision Making:      

#                      if person_detected and..... ==> this working as an AND gate , to take a pic and save it should be cought someone and 5 sec passed at least

#                      now = datetime.now() ==> to get the time and the date now 

#                      cv2.imwrite(image_path, frame) ==> it takes the current frame and save it to the path we selected

#                      with open("log.txt", "a", encoding="utf-8") as log_file ==> it opens the txt file and 'a' (append) 

#                      last_save_time = current_time ==> Updating the last save time 

#                      cv2.imshow("Smart Security Camera", frame) ==> this shows the frame live , it's actuallu not a video it's just so fast pics (30 pic/sec)

#                      if cv2.waitKey(1) & 0xFF == ord('q'):  break ==> this function waits 1ms to see if you pressed 'Q' or not , is yes it will break the loop 

#========================================================================================================================================================================================================

# cleaning :           cap.release() ==> turns off the cam 
#                      cv2.destroyAllWindows() ==> closes the show window and clear it from the RAM






while True:
    # 1. Read Camera
    ret, frame = cap.read()    
    if not ret:
        print("Error: Could not read from the camera!")
        break

    # 2. AI Detection
    results = model(frame, classes=[0], conf=0.5, verbose=False) 
    person_detected = False

    # 3. Draw Boxes
    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(frame, "Intruder!", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            person_detected = True

    # 4. Decision Making
    current_time = time.time()
    if person_detected and (current_time - last_save_time) > cooldown_seconds:
        now = datetime.now()
        time_str = now.strftime("%Y-%m-%d_%H-%M-%S")
        
        # Save the image
        image_path = f"Intruders/intruder_{time_str}.jpg"
        cv2.imwrite(image_path, frame)
        
        # Log the event
        with open("log.txt", "a", encoding="utf-8") as log_file:
            log_file.write(f"⚠ Intruder detected in frame at: {now.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
        print(f"⚠ Alert: Image of the intruder captured and saved as {image_path}")
        last_save_time = current_time

    # 5. Display the live video feed
    cv2.imshow("Smart Security Camera", frame)

    # 6. Exit when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Clean up memory and release the camera
cap.release()
cv2.destroyAllWindows()